In [ ]:
import numpy as np
from scipy.stats import random_correlation, wishart, beta, gamma
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:
def plot_correlation_heatmap(correlation_matrix):
    fig, ax = plt.subplots()
    heatmap = ax.imshow(correlation_matrix, cmap='RdYlBu_r')
    cbar = plt.colorbar(heatmap)
    cbar.set_label('Correlation')

    # Set title and axis labels
    ax.set_title("Correlation Matrix Heatmap")

    # Show the plot
    plt.tight_layout()
    plt.show()

### Eigenvalue Method

In [ ]:
def generate_random_eigenvalues(N, shape, scale):
    random_values = gamma.rvs(shape, scale=scale, size=N)
    random_eigenvalues = random_values / np.sum(random_values)  # Normalize to sum up to 1
    return random_eigenvalues

In [ ]:
N = 100    # Number of eigenvalues
shape = 1  # Shape parameter of the gamma distribution
scale = 1  # Scale parameter of the gamma distribution
eigenvalues = generate_random_eigenvalues(N, shape, scale)
eigenvalues = np.flip(np.sort(eigenvalues))*N

In [ ]:
rng = np.random.default_rng()
ev_corr =  random_correlation.rvs(eigenvalues, random_state=rng)

In [ ]:
plot_correlation_heatmap(ev_corr)

### Random Gram Method

In [ ]:
def generate_random_vectors(n, m):
    """
    Generate n random vectors on the unit sphere in m dimensions.
    """
    vectors = np.random.normal(size=(n, m))
    # Normalize each vector
    vectors /= np.linalg.norm(vectors, axis=1)[:, np.newaxis]
    return vectors

In [ ]:
def generate_gram_correlation_matrix(n, m):
    """
    Generate a random correlation matrix using the Gram method.
    """
    # Generate n random vectors on the unit sphere in m dimensions
    U = generate_random_vectors(n, m)
    # Compute the Gram matrix
    C = np.dot(U, U.T)
    return C

In [ ]:
g_corr = generate_gram_correlation_matrix(N, 2*N)

In [ ]:
plot_correlation_heatmap(g_corr)

### Wishart Distribution

In [ ]:
def generate_wishart_matrix(n, df):
    """
    Generate a random matrix from the Wishart distribution.
    """
    # Define the scale matrix as the identity matrix
    scale = np.eye(n)
    # Sample a matrix from the Wishart distribution
    W = wishart.rvs(df, scale)
    return W

In [ ]:
def generate_wishart_correlation_matrix(W):
    """
    Generate a correlation matrix from a Wishart matrix.
    """
    # Compute the standard deviations
    std_devs = np.sqrt(np.diag(W))
    # Normalize the Wishart matrix to get the correlation matrix
    C = W / np.outer(std_devs, std_devs)
    return C

In [ ]:
# Generate a random matrix from the Wishart distribution
W = generate_wishart_matrix(N, N)

In [ ]:
w_corr = generate_wishart_correlation_matrix(W)

In [ ]:
plot_correlation_heatmap(w_corr)

### Onion

In [ ]:
def generate_onion_correlation_matrix(d):
    S = np.array([[1]])
    for k in range(2, d+1):
        if k == d:
            continue
        y = beta.rvs((k-1)/2, (d-k)/2)  # sampling from beta distribution
        r = np.sqrt(y)
        theta = np.random.randn(k-1)
        theta = theta / np.linalg.norm(theta)
        w = r * theta
        E, U = np.linalg.eig(S)
        R = U @ np.diag(np.sqrt(E)) @ U.T  # R is a square root of S
        q = R @ w
        S = np.block([[S, q[:, None]], [q[None, :], 1]])  # increasing the matrix size
    return S

In [ ]:
o_corr = generate_onion_correlation_matrix(N)

In [ ]:
plot_correlation_heatmap(o_corr)

### Extended Onion

In [ ]:
def generate_ext_onion_correlation(d, eta):
    b = eta + (d-2)/2
    u = beta.rvs(b, b)
    r12 = 2*u - 1
    S = np.array([[1, r12], [r12, 1]])

    for k in range(3, d+1):
        b = b - 1/2
        y = beta.rvs((k-1)/2, b)
        r = np.sqrt(y)
        theta = np.random.randn(k-1)
        theta = theta / np.linalg.norm(theta)
        w = r * theta
        E, U = np.linalg.eig(S)
        R = U @ np.diag(np.sqrt(E)) @ U.T  # R is a square root of S
        q = R @ w
        S = np.block([[S, q[:, None]], [q[None, :], 1]])  # increasing the matrix size
    return S

In [ ]:
eo_corr = generate_ext_onion_correlation(N, 1)

In [ ]:
plot_correlation_heatmap(eo_corr)

### Vines Copula

In [ ]:
def generate_vine_correlation(d, eta):
    b = eta + (d-1)/2
    P = np.zeros((d, d))  # storing partial correlations
    S = np.eye(d)

    for k in range(d-1):
        b = b - 1/2
        for i in range(k+1, d):
            P[k, i] = beta.rvs(b, b)  # sampling from beta
            P[k, i] = (P[k, i] - 0.5) * 2  # linearly shifting to [-1, 1]
            p = P[k, i]
            for l in range(k-1, -1, -1):  # converting partial correlation to raw correlation
                p = p * np.sqrt((1 - P[l, i]**2) * (1 - P[l, k]**2)) + P[l, i] * P[l, k]
            S[k, i] = p
            S[i, k] = p
    return S

In [ ]:
v_corr = generate_vine_correlation(N, 1)

In [ ]:
plot_correlation_heatmap(v_corr)

In [ ]:
def plot_correlation_heatmaps(correlation_matrices, labels, filename, C):
    N = len(correlation_matrices)
    R = int(np.ceil(N / C))  # calculate the number of rows needed

    fig, axs = plt.subplots(R, C, figsize=(C*5, R*5))  # adjust the size of the figure
    axs = axs.ravel()  # flatten the axes array

    for i, (corr_mat, label) in enumerate(zip(correlation_matrices, labels)):
        heatmap = axs[i].imshow(corr_mat, cmap='RdYlBu_r')
        axs[i].set_title(label)

    # Hide unused subplots
    if N < R*C:
        for i in range(N, R*C):
            axs[i].axis('off')
            

    # Add a single colorbar
    cbar = fig.colorbar(heatmap, ax=axs, orientation='horizontal', pad=0.06)
    cbar.set_label('Correlation')

    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
c_lst = [ev_corr, g_corr, w_corr, o_corr, eo_corr, v_corr]
nm_lst = ['Eigenvalue', 'Random Gram', 'Wishart', 'Onion', 'Extended Onion', 'Vine Copula']

In [ ]:
C = 2 
filename = 'correlation_heatmaps.png'
plot_correlation_heatmaps(c_lst, nm_lst, filename, C)

In [ ]:
for c in c_lst:
    np.linalg.cholesky(c)